In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from fundus_data_toolkit.functional import open_image
from jppype import Mosaic, vscode_theme

from fundus_odmac_toolkit.models.segmentation import segment
from fundus_toolkits import FundusData
from fundus_vessels_toolkit import VTree
from fundus_vessels_toolkit.models import segment_av
from fundus_vessels_toolkit.pipelines.avseg_to_tree import GNNAVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.segment_to_graph.tree_topology import TreeTopology, optimal_lines
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from fundus_vessels_toolkit.utils.jppype import draw_graph, draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

## Load Image and Segment AV, OD, Macula


In [ ]:
PATH = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/MAPLES-DR/")
RAW = PATH / "1-images"
AV = PATH / "2-av"
TOPO = PATH / "3-topo"
IMG = sorted(list(RAW.glob("*.png")))[5].stem

fundus_gt = FundusData(image=RAW / (IMG + ".png"), av=AV / (IMG + ".png"))
trees_gt = VTree.load(TOPO / f"{IMG}_art.npz"), VTree.load(TOPO / f"{IMG}_vei.npz")
topo_gt = (
    TreeTopology.from_tree(trees_gt[0], expand_labels_by=5),
    TreeTopology.from_tree(trees_gt[1], expand_labels_by=5),
)


od_mac = segment(open_image(RAW / (IMG + ".png"))).numpy(force=True).argmax(axis=0)
fundus_gt = fundus_gt.update(od=od_mac == 1, macula=od_mac == 2, reshape_method="resize")
fundus = fundus_gt.copy()
_ = segment_av(fundus)

print(IMG)

20051020_58065_0100_PP


In [20]:
av2tree = GNNAVSegToTree()

trees = av2tree(fundus)

graph = av2tree.to_vgraph(fundus).sort_branches_by_nodesID()
digraph = VBranchDigraph.from_graph(graph, max_distance=150)
digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])

m = Mosaic(3, cols_titles=["Predicted", "Predicted Prepared", "Ground Truth"], cell_height=600, background=fundus.image)
fundus.draw(view=m[0])
draw_graph(graph, view=m[0], edge_labels=True, node_labels=True)
fundus.draw(view=m[1])
draw_tree(
    digraph.optimize_tree(keep_invalid_branch=True),
    view=m[1],
    branch_color="subtree",
    edge_labels=True,
    node_labels=True,
    bspline_dir=True,
)
fundus_gt.draw(view=m[2])
draw_trees(trees_gt, view=m[2])
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

In [21]:
pd.DataFrame(digraph.lines_by_branch(112))

,0,1,2,3,4,5,6
0,-1.0,225.0,112.0,130.0,1.0,0.997287,0.997527
1,115.0,130.0,112.0,130.0,0.0,0.947294,0.997527
2,112.0,130.0,113.0,130.0,0.0,0.002473,0.997527
3,113.0,130.0,112.0,130.0,0.0,0.002473,0.997527
4,112.0,130.0,115.0,130.0,0.0,0.002473,0.052706
5,-1.0,225.0,112.0,74.0,0.0,0.997287,0.002473


In [22]:
np.argwhere(digraph.invalid_branch()).flatten()

array([  4,   5,  16,  24,  42,  50,  68,  73,  77,  94, 107, 122, 124,
       129, 174, 176, 179, 181, 183, 186, 188])

In [25]:
digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])

In [8]:
digraph.optimize_tree()

/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/segment_to_graph/vbranch_digraph.py:172: UserWarning: Removed 20 invalid branches from the optimized tree.
  warnings.warn(f"Removed {B_inv} invalid branches from the optimized tree.", UserWarning)


In [9]:
digraph.optimize_tree()

/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/segment_to_graph/vbranch_digraph.py:172: UserWarning: Removed 20 invalid branches from the optimized tree.
  warnings.warn(f"Removed {B_inv} invalid branches from the optimized tree.", UserWarning)


In [10]:
%timeit digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])

27.3 ms ± 1.38 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [11]:
%timeit VBranchDigraph.from_graph(graph, max_distance=300)

68.2 ms ± 1.59 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [12]:
from fundus_vessels_toolkit.segment_to_graph.geometry_parsing import derive_tips_geometry_from_curve_geometry
from fundus_vessels_toolkit.segment_to_graph.graph_simplification import find_facing_tips
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import prepare_graph_for_reconnections
from fundus_vessels_toolkit.vascular_data_objects.vbranch_geodata import VBranchGeoData


g = graph.copy()
g.geometric_data().clear_attribute(all_except={VBranchGeoData.Fields.TANGENTS, VBranchGeoData.Fields.TIPS_TANGENT})
%timeit prepare_graph_for_reconnections(g, max_distance=300, max_angle=30, inplace=False)
prepare_graph_for_reconnections(g, max_distance=300, max_angle=30, inplace=True)
%timeit derive_tips_geometry_from_curve_geometry(graph, tangent=True, inplace=False)
derive_tips_geometry_from_curve_geometry(graph, tangent=True, inplace=True)

%timeit find_facing_tips(graph, max_distance=300)

23.9 ms ± 1.59 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
4.64 ms ± 160 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
15 ms ± 111 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [13]:
def draw_cone(branch_id: int, first_tip: bool, view=None, pos_tolerance=15, max_dist=100, max_angle=30):
    sqr_max_dist = max_dist * max_dist
    sqr_pos_tolerance = pos_tolerance * pos_tolerance

    min_cos = np.cos(np.deg2rad(max_angle))

    geodata = graph.geometric_data()
    tips_pos = (
        geodata.tip_coord(branch_id=branch_id, first_tip=first_tip).astype(np.float64).reshape(-1, 2)
    )  # [branch_id x (tip0, tip1), (y,x)]
    tips_tan = geodata.tip_tangent(branch_id=branch_id, first_tip=first_tip).reshape(
        -1, 2
    )  # [branch_id x (tip0, tip1), (y,x)]

    yy, xx = np.meshgrid(np.arange(fundus.image.shape[1]), np.arange(fundus.image.shape[2]), indexing="ij")
    yx = np.stack((yy, xx), axis=-1).reshape(-1, 2)

    tips_dtan = tips_pos[:, None, :] - yx[None, :, :]  # (tip_origin, tip_destination, yx)
    tips_dsqr = np.square(tips_dtan).sum(axis=2)
    tips_dtan /= np.sqrt(tips_dsqr)[..., None] + 1e-8

    # === VICINITY CHECK ===
    # Given a tip p0 with tangent t0 (oriented towards its curve)
    # we define a cone oriented towards -t0 with apex at p0 + t0 * pos_tolerance (so the tip itself is inside the cone)
    # and opening angle max_angle at distance pos_tolerance and 60 degrees at distance 0 from the apex.
    apex = tips_pos + tips_tan * pos_tolerance  # Cone apex position
    apex2tips = apex[:, None, :] - yx[None, :, :]
    apex2tips_dsqr = np.square(apex2tips).sum(axis=2)
    apex2tips /= np.sqrt(apex2tips_dsqr)[..., None] + 1e-8
    apex_cos = (tips_tan[:, None, :] * apex2tips).sum(axis=2)
    inside_cone = ((apex_cos >= min_cos) | (tips_dsqr <= sqr_pos_tolerance)) & (tips_dsqr <= sqr_max_dist)
    yx = yx[inside_cone[0]]
    map = np.zeros(fundus.image.shape[1:], dtype=np.uint8)
    map[yx[:, 0], yx[:, 1]] = 1
    if view is not None:
        view.add_label(map, "cone", opacity=0.2)

In [14]:
yy, xx = np.meshgrid(np.arange(fundus.image.shape[1]), np.arange(fundus.image.shape[2]), indexing="ij")
yx = np.stack((yy, xx), axis=-1).reshape(-1, 2)
yx


array([[   0,    0],
       [   0,    1],
       [   0,    2],
       ...,
       [1499, 1497],
       [1499, 1498],
       [1499, 1499]], shape=(2250000, 2))

In [15]:
from fundus_vessels_toolkit.segment_to_graph.graph_simplification import find_facing_tips

find_facing_tips(digraph.graph)

array([[[[False, False],
         [False, False],
         [False, False],
         ...,
         [False, False],
         [False, False],
         [False, False]],

        [[False, False],
         [False,  True],
         [False, False],
         ...,
         [False, False],
         [False, False],
         [False, False]]],


       [[[False, False],
         [False, False],
         [False, False],
         ...,
         [False, False],
         [False, False],
         [False, False]],

        [[False,  True],
         [False, False],
         [False, False],
         ...,
         [False, False],
         [False, False],
         [False, False]]],


       [[[False, False],
         [False, False],
         [False, False],
         ...,
         [False, False],
         [False, False],
         [False, False]],

        [[False, False],
         [False, False],
         [False, False],
         ...,
         [False, False],
         [False, False],
         [False, False]]],


In [16]:
optimal_lines(digraph.graph, topo_gt[0], digraph.line_list)

(array([False, False, False, ..., False, False, False], shape=(2006,)),
 array([ 0.        ,  1.        , -0.94736842,  0.        ,  0.        ,
        -1.        ,  1.        , -1.        ,  1.        ,  0.        ,
        -1.        , -1.        ,  0.        ,  0.        , -1.        ,
         1.        ,  1.        , -1.        ,  1.        ,  1.        ,
        -1.        ,  1.        ,  1.        , -1.        , -1.        ,
         0.        , -1.        , -1.        , -1.        ,  0.        ,
        -1.        , -1.        ,  0.        ,  0.        , -1.        ,
         0.        ,  0.        ,  0.        ,  1.        ,  0.        ,
         0.        , -1.        ,  0.        ,  0.        , -1.        ,
        -1.        ,  0.        ,  0.97142857,  0.        ,  0.        ,
         0.        , -1.        ,  0.        ,  0.        ,  0.        ,
         0.        , -1.        , -1.        ,  0.        ,  1.        ,
         0.        ,  0.        , -1.        ,  1.  

## Compute AV topological maps


In [17]:
from fundus_vessels_toolkit.segment_to_graph.av_map_fixing import TopologicalLabel, rasterize_tree_topology

topo_maps = [rasterize_tree_topology(tree, expand_labels_by=10) for tree in trees_gt]
(art_labels, art_topo), (vei_labels, vei_topo) = topo_maps

m = Mosaic(
    (2, 3),
    cols_titles=["VTree", "Branch labels", "Topology map"],
    rows_titles=["Art.", "Vein"],
    cell_height=800,
    background=fundus.image,
)
draw_tree(trees_gt[0], view=m[0, 0], artery=True, edge_labels=False)
m[0, 0].add_label(fundus_gt.av == 1, colormap="red", opacity=0.2)
m[0, 1].add_image(TopologicalLabel.map_to_rgb(art_labels))
m[0, 2].add_image(np.repeat(art_topo[:, :, None], 3, axis=2))
fundus_gt.draw(view=m[1, 0])
draw_tree(trees_gt[1], view=m[1, 0], artery=False, edge_labels=False)
m[1, 1].add_image(TopologicalLabel.map_to_rgb(vei_labels))
m[1, 2].add_image(np.repeat(vei_topo[:, :, None], 3, axis=2))
m

ImportError: cannot import name 'TopologicalLabel' from 'fundus_vessels_toolkit.segment_to_graph.av_map_fixing' (/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/segment_to_graph/av_map_fixing.py)

In [ ]:
m = Mosaic(
    3,
    cell_height=400,
    background=fundus.image,
)
fundus_gt.draw(view=m[0])
draw_trees(trees, view=m[0], edge_labels=True, bspline_dir=True)
m[1].add_image(TopologicalLabel.map_to_rgb(art_labels))
draw_tree(trees[0], view=m[1], artery=True)
m[2].add_image(np.repeat(vei_topo[:, :, None], 3, axis=2))
draw_tree(trees[1], view=m[2], artery=False)
m

In [ ]:
import pandas as pd
from fundus_vessels_toolkit.segment_to_graph.av_map_fixing import evaluate_topology

df = pd.DataFrame(evaluate_topology(trees[0], art_labels, art_topo))
df

In [ ]:
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import naive_infer_roots
from fundus_vessels_toolkit.segment_to_graph.models.training import deteriorate_segmentation


av = deteriorate_segmentation(
    fundus_gt,
)
fundus_gt2 = fundus_gt.update(av=av)
trees_gt2 = naive_av2tree(fundus_gt2)
art_graph, candidates = prepare_graph_for_reconnections(
    trees_gt2[0], max_distance=300
)  # , branch_ids=[25], endpoint_ids=[84])
art_tree = naive_infer_roots(art_graph, fundus_gt2.od_center)

m = Mosaic(2, cols_titles=["Predicted", "Ground Truth"], cell_height=800)
fundus_gt2.draw(view=m[0])
draw_tree(art_tree, artery=True, view=m[0])

m[1].add_image(TopologicalLabel.map_to_rgb(art_labels))
draw_trees(trees_gt2, view=m[1])
m

In [ ]:
from fundus_vessels_toolkit.segment_to_graph.line_digraph_solving import prepare_graph_for_reconnections
from fundus_vessels_toolkit.utils.jppype import draw_graph

m = Mosaic(2, cols_titles=["Predicted", "Ground Truth"], cell_height=800)
fundus_gt.draw(view=m[0])
draw_tree(trees_gt2[0], edge="skeleton", artery=True, view=m[0], edge_labels=True)
fundus_gt.draw(view=m[1])
_, candidates = prepare_graph_for_reconnections(trees_gt2[0], max_distance=300)  # , branch_ids=[25], endpoint_ids=[84])
draw_graph(_, view=m[1], node_labels=True)
m

In [ ]:
_.node_coord()[25]

In [ ]:
candidates[candidates[:, 1] == 107]

In [ ]:
df_art = pd.DataFrame(evaluate_topology(trees_gt2[0], art_labels, art_topo))
df_vei = pd.DataFrame(evaluate_topology(trees_gt2[1], vei_labels, vei_topo))

In [ ]:
df_art.iloc[48]